<a href="https://colab.research.google.com/github/moazamzf/code-switching-codesaviours-si26--Moazam-/blob/main/SI26-Week7-moazam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch datasets seqeval -q

import pandas as pd
from sklearn.model_selection import train_test_split

# Load your dataset
df = pd.read_csv('Week6-upd.csv')

# Sanity check BEFORE grouping — catch label typos now, not after training starts
print(df.head())
print(df['label'].value_counts())

# Label mapping — must exactly match the unique values in df['label']
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

# Confirm no unexpected labels slipped in
unexpected = set(df['label'].unique()) - set(label2id.keys())
if unexpected:
    raise ValueError(f"Unexpected labels found: {unexpected} — fix these in dataset.csv before continuing")

# Group by sentence into word/label lists (safer than .apply returning dicts)
grouped = df.groupby('sentence').agg(list)
sentences = [
    {'words': row['word'], 'labels': row['label']}
    for _, row in grouped.iterrows()
]

# Split into train and test
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
                              sentence  word label
0  Aaj mera mood nahi hai for anything   Aaj   URD
1  Aaj mera mood nahi hai for anything  mera   URD
2  Aaj mera mood nahi hai for anything  mood   URD
3  Aaj mera mood nahi hai for anything  nahi   URD
4  Aaj mera mood nahi hai for anything   hai   URD
label
URD    833
ENG    427
MIX     36
Name: count, dtype: int64
Training sentences: 120
Testing sentences: 30


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import Dataset

model_name = 'xlm-roberta-base'

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    eval_strategy='epoch',       # renamed from evaluation_strategy in newer transformers
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,   # use tokenizer=tokenizer if this errors on older transformers
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

print('Starting training...')
trainer.train()
print('Training complete!')


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [ ]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch
0.255535,0.198515,5


{'eval_loss': 0.19851499795913696}


In [ ]:
import numpy as np
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(test_ds)
predictions = np.argmax(predictions, axis=2)

true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
true_predictions = [
    [id2label[p] for p, l in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

              precision    recall  f1-score   support

          IX       0.00      0.00      0.00         6
          NG       0.90      0.98      0.94        96
          RD       0.96      0.92      0.94        72

   micro avg       0.92      0.92      0.92       174
   macro avg       0.62      0.63      0.63       174
weighted avg       0.89      0.92      0.91       174



In [ ]:
from sklearn.metrics import classification_report as sk_report

flat_true = [l for seq in true_labels for l in seq]
flat_pred = [p for seq in true_predictions for p in seq]

print(sk_report(flat_true, flat_pred, digits=3))

              precision    recall  f1-score   support

         ENG      0.904     0.979     0.940        96
         MIX      0.000     0.000     0.000         6
         URD      0.987     0.974     0.980       154

    accuracy                          0.953       256
   macro avg      0.630     0.651     0.640       256
weighted avg      0.933     0.953     0.942       256



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a HF token with WRITE access when prompted

In [ ]:
repo_name = 'code-switching-codesaviours-si26-moazam'

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f'Model published at: https://huggingface.co/Moazamzf/{repo_name}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g7xcuh6/model.safetensors:   1%|          | 7.32MB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6ihef2a8/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model published at: https://huggingface.co/Moazamzf/code-switching-codesaviours-si26-moazam
